In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

In [2]:
def wait_time_t2(i,j,k,L1, L2, R1, c1, R2, c2, mu):
    n_star = math.floor(mu * ((R1) / c1))
    T = 0
    p = L1 / mu
    n = i + k
    K = n_star + 1
    t1 = (1/mu)*(1/(1-p)**2)*( n*(1-p) - (1-p**n)*p**(K-n+1) )

    m = 1
    t2 = (1/mu)*(1/(1-p)**2)*( m*(1-p) - (1-p**m)*p**(K-m+1) )

    T = t1 + t2*j

    return T

In [3]:
def util2(i,j,k,L1, L2, R1, c1, R2, c2, mu):
    util = R2 - c2*wait_time_t2(i,j,k,L1, L2, R1, c1, R2, c2, mu)
    return util

In [4]:
def will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, mu):
    U = util2(i,j,k,L1, L2, R1, c1, R2, c2, mu)

    if U >=0:
        return 1
    else:
        return 0

In [5]:
def find_space(L1, L2, R1, c1, R2, c2, mu):
    n_star = math.floor(mu * ((R1) / c1))
    state_space = [(i, 0, 1) for i in range(n_star+1)]
    x = 0
    y = len(state_space)
    
    while x < y:
        if will_enter_t2_state_dependent(state_space[x][0], state_space[x][1], state_space[x][2],L1, L2, R1, c1, R2, c2,  mu) == 1:
            state_space.append( ( state_space[x][0], state_space[x][1] + 1, state_space[x][2] ) )
    
        y = len(state_space)
        x += 1
    
    for s in state_space:
        a,b,c = s
        for i in range(n_star+1):
            if (i,b,c) not in state_space:
                state_space.append((i,b,c))
    
    
    state_space = [(0,0,0)] + state_space
    
    return state_space

In [6]:
def solution(L1, L2, R1, c1, R2, c2, mu):
    # TYPE 1
    n_star = math.floor(mu * ((R1) / c1))
    if n_star < 0:
        n_star = 0

    #Type-1 steady-state probs. using the formulas from type-1 Markov chain
    type1_probs = {}
    π0 = (mu - L1) / (mu + L2 - (L1 + L2) * ((L1 / mu) ** (n_star+1)))
    type1_probs[0] = π0
    for n in range(1, n_star + 2):
        π_n = π0 * ((L1 / mu) ** (n - 1)) * ((L1 + L2) / mu)
        type1_probs[n] = π_n
    result1 = type1_probs

    #Type-1 Joining rate and equilibrium lambda
    eq_L1 = L1*(1-result1[(n_star+1)])

    h1 = 1-result1[(n_star+1)]


    # TYPE 2
    states = find_space(L1, L2, R1, c1, R2, c2, mu)
    total_states = len(states)
    transition_matrix = np.zeros((total_states, total_states))

    for idx, (i, j, k) in enumerate(states):
        current_state = idx

        if k==0:
            next_state = states.index((0, 0, 1)) 
            transition_matrix[current_state, next_state] = L1+L2      
            
        if k!=0 and i==0 and j==0:
            next_state = states.index((0,0,0))
            transition_matrix[current_state, next_state] = mu
            
            if k+i<=n_star:
                next_state = states.index((1,0,1))
                transition_matrix[current_state, next_state] = L1
            
            if will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, mu) == 1:
                next_state = states.index((0,1,1))
                transition_matrix[current_state, next_state] = L2   
                         
        if j==0 and i!=0:
            next_state = states.index((i-1, j, 1)) if (i-1, j, 1) in states else None
            if next_state is not None:
                transition_matrix[current_state, next_state] = mu
            
            if k+i<=n_star:
                next_state = states.index((i+1, j, 1)) if (i+1, j, 1) in states else None
                if next_state is not None:
                    transition_matrix[current_state, next_state] = L1
            
            if will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, mu) == 1:
                next_state = states.index((i, j+1, 1)) if (i, j+1, 1) in states else None
                if next_state is not None:
                    transition_matrix[current_state, next_state] = L2

        if j!=0 and i==0:
            next_state = states.index((i, j-1, 1)) if (i, j-1, 1) in states else None
            if next_state is not None:
                transition_matrix[current_state, next_state] = mu
                
            if k+i<=n_star:
                next_state = states.index((i+1, j, 1)) if (i+1, j, 1) in states else None
                if next_state is not None:
                    transition_matrix[current_state, next_state] = L1
            
            if will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, mu) == 1:
                next_state = states.index((i, j+1, 1)) if (i, j+1, 1) in states else None
                if next_state is not None:
                    transition_matrix[current_state, next_state] = L2
        
        if j!=0 and i!=0:
            next_state = states.index((i-1, j, 1)) if (i-1, j, 1) in states else None
            if next_state is not None:
                transition_matrix[current_state, next_state] = mu
    
            if k+i<=n_star:
                next_state = states.index((i+1, j, 1)) if (i+1, j, 1) in states else None
                if next_state is not None:
                    transition_matrix[current_state, next_state] = L1
            
            if will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, mu) == 1:
                next_state = states.index((i, j+1, 1)) if (i, j+1, 1) in states else None
                if next_state is not None:
                    transition_matrix[current_state, next_state] = L2
            
    #Creating Q Matrix and solving for steady-state probs. of type-2 markov chain
    #https://vknight.org/blog/posts/continuous-time-markov-chains/
    n = transition_matrix.shape[0]
    adjusted_matrix = transition_matrix.copy()
    for i in range(n):
        adjusted_matrix[i, i] = -np.sum(transition_matrix[i, :])    
    Q=adjusted_matrix
    dimension = Q.shape[0]
    M = np.vstack((Q.transpose()[:-1], np.ones(dimension)))
    b = np.vstack((np.zeros((dimension - 1, 1)), [1]))
    state=np.linalg.solve(M, b).transpose()[0]
    type2_probs=[]
    for i in state:
        type2_probs.append(float(i))
    result_dict = dict(zip(states, type2_probs))
    
    #Type-2 joining rate equilibrium lambda
    type2_joining_rate = 0
    for state in states:
        if will_enter_t2_state_dependent(state[0], state[1],state[2],L1, L2, R1, c1, R2, c2, mu) == 1:
            type2_joining_rate += result_dict[state]
                  
    eq_L2 = L2*type2_joining_rate
    h2 = type2_joining_rate
    result2 = result_dict # result of type 2
    
    return h1, h2, result1, result2

In [7]:
def metrics(L1, L2, R1, c1, R2, c2, mu):
    h1, h2 , pi1, pi2 = solution(L1, L2, R1, c1, R2, c2, mu)
    state1 = list(pi1.keys())
    state2 = list(pi2.keys())

    ### W1
    W1 = 0
    for s in state1[:-1]:
        W1 += (s/mu)*pi1[s]


    ### W2
    W2 = 0
    for s in state2:
        i,j,k = s[0], s[1], s[2]
        W2 += wait_time_t2(i,j,k,L1, L2, R1, c1, R2, c2, mu)*pi2[s]*will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, mu)


    ### Utility 1
    U1 = 0
    for s in state1[:-1]:
        W1_now = (s/mu)
        U1 += pi1[s]*(R1 - c1*W1_now)


    ### Utility 2
    U2 = 0
    for s in state2:
        i,j,k = s[0], s[1], s[2]
        W2_now = wait_time_t2(i,j,k,L1, L2, R1, c1, R2, c2, mu)
        U2 += pi2[s]*(R2 - c2*W2_now)*will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, mu)


    ### Throughput
    thr = (L1*h1 + L2*h2)/(L1+L2)

    
    ### Cumulative Utility 

    cum_U = U1*h1*L1 + U2*h2*L2

    
    dic = {(L1, L2, R1, c1, R2, c2, mu): (h1, h2, W1, W2, U1, U2, thr, cum_U) }


    return dic 